# Actividad 8 — Monitorización de IA en Google Colab
**Proyecto:** control de acceso vehicular (simulación sin barreras físicas).

Este notebook descarga el repositorio público, ejecuta pruebas y dos incidentes sintéticos, genera indicadores y exporta evidencias. **No presenta gráficos de Colab como si fueran Grafana.**


In [ ]:
import os, subprocess, pathlib
ROOT = pathlib.Path('/content/Actividad8-Monitorizacion-IA-Control-Acceso-Vehicular')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/roquegarciaespinoza-ops/Actividad8-Monitorizacion-IA-Control-Acceso-Vehicular.git',str(ROOT)],check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'],check=True)
os.chdir(ROOT)
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
import subprocess
subprocess.run(['python','-m','unittest','discover','-s','tests','-v'],check=True)


In [ ]:
import json, pathlib
from simulations.incident_01_latency import run as latency_run
from simulations.incident_02_drift import run as drift_run
out=pathlib.Path('evidence/generated');out.mkdir(parents=True,exist_ok=True)
latency_result=latency_run();drift_result=drift_run()
for name,result in [('incident_01_latency',latency_result),('incident_02_drift',drift_result)]:
    (out/(name+'.json')).write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
    print(name, result['status'])


In [ ]:
import time, statistics, matplotlib.pyplot as plt
from src.monitoring import infer, prometheus_metrics
scenarios=[('Base',15),('Latencia degradada',300),('Recuperación',15)]
observations=[]
for name,delay in scenarios:
    values=[]
    for _ in range(8):
        t=time.perf_counter();result=infer(simulated_latency_ms=delay)
        assert result['access_granted'] is False
        values.append((time.perf_counter()-t)*1000)
    observations.append({'escenario':name,'media_ms':statistics.mean(values),'max_ms':max(values),'muestras':len(values)})
print(json.dumps(observations,indent=2,ensure_ascii=False))
(out/'latency_observations.json').write_text(json.dumps(observations,indent=2,ensure_ascii=False),encoding='utf-8')
fig,ax=plt.subplots(figsize=(9,4));ax.bar([x['escenario'] for x in observations],[x['media_ms'] for x in observations]);ax.set_ylabel('Latencia media observada (ms)');ax.set_title('Simulación: degradación y recuperación de latencia');fig.tight_layout();fig.savefig(out/'latency_chart.png',dpi=150);plt.show()
(out/'prometheus_metrics.txt').write_text(prometheus_metrics(),encoding='utf-8')


In [ ]:
from src.drift_detection import confidence_shift
ref=[0.96,0.97,0.95,0.98,0.96,0.97]
shift=[0.39,0.44,0.42,0.40,0.43,0.41]
recovered=ref.copy()
fig,ax=plt.subplots(figsize=(8,4));ax.plot(ref,label='Referencia');ax.plot(shift,label='Degradación sintética');ax.plot(recovered,label='Recuperación sintética');ax.set_ylim(0,1);ax.set_ylabel('Confianza simulada');ax.set_xlabel('Observación');ax.legend();ax.set_title('Señal didáctica: cambio de confianza (no demuestra data drift)');fig.tight_layout();fig.savefig(out/'confidence_chart.png',dpi=150);plt.show()
print(confidence_shift(ref,shift))


In [ ]:
import zipfile, pathlib
archive=pathlib.Path('/content/Actividad8_evidencia_colab.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in out.glob('*'):
        if p.is_file():z.write(p,arcname='evidencia_colab/'+p.name)
print('Evidencia creada:',archive)
from google.colab import files
files.download(str(archive))
